# AKT26 Homework 4 - GPU validation and benchmarks

Team HaT - Session 4: rank-1 update and small batched GEMM.

Select **Runtime > Change runtime type > GPU**, then run the cells in order.
This notebook builds the submitted CUDA C++ files, runs correctness tests,
compares both assignments with cuBLAS, and exports actual measured results.
No GPU execution results are pre-filled.

Use the supplied `AKT26_HW4_CUDA_Source.zip`, or enter your published repository
URL in the source cell. The original `3-ger` and `4-gemm` paths are retained.


In [ ]:
import os
import shutil
import subprocess
import tempfile
import zipfile
from pathlib import Path

subprocess.run(["nvidia-smi"], check=True)
NVCC = shutil.which("nvcc")
if NVCC is None:
    raise RuntimeError("nvcc is unavailable. Select a Colab GPU runtime.")
os.environ["NVCC"] = NVCC
subprocess.run([NVCC, "--version"], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is available in this runtime.")
major, minor = torch.cuda.get_device_capability()
SM = f"{major}{minor}"
print("GPU:", torch.cuda.get_device_name(), "target sm_" + SM)


## Load the completed source

The default URL loads the published `atta-hilali/CUDA` repository.
Set `REPOSITORY_URL` to an empty string to upload the source ZIP instead. Each run uses a new working folder.


In [ ]:
REPOSITORY_URL = "https://github.com/atta-hilali/CUDA.git"
workspace = Path(tempfile.mkdtemp(prefix="akt26-hw4-", dir="/content"))
if REPOSITORY_URL:
    repo = workspace / "akt26_gpu_dev"
    subprocess.run(["git", "clone", "--", REPOSITORY_URL, str(repo)], check=True)
else:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one source ZIP archive.")
    archive_name, archive_data = next(iter(uploaded.items()))
    import io
    with zipfile.ZipFile(io.BytesIO(archive_data)) as archive:
        for member in archive.infolist():
            target = (workspace / member.filename).resolve()
            if not target.is_relative_to(workspace.resolve()):
                raise ValueError("Unsafe archive path")
        archive.extractall(workspace)
    matches = [p.parent.parent for p in workspace.rglob("3-ger/ger.cu")]
    if len(matches) != 1:
        raise RuntimeError("Could not locate the assignment root in the ZIP.")
    repo = matches[0]
assert (repo / "4-gemm/gemm.cu").is_file()
os.chdir(repo)
print("Source:", repo)


## Compile and test

The test executable includes the submitted kernels directly. It covers odd
sizes, all `n=1..32`, independent batches, zero/identity/ones inputs, and guarded
output allocations. Both assignment drivers also check their cuBLAS results.
A failed command stops the notebook; do not report it as a successful run.


In [ ]:
results = repo / "results"
results.mkdir(exist_ok=True)
def run_logged(command, filename):
    completed = subprocess.run(command, text=True, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, cwd=repo)
    output = "$ " + " ".join(map(str, command)) + "\n" + completed.stdout
    (results / filename).write_text(output)
    print(output)
    completed.check_returncode()

cuda_dir = str(Path(NVCC).resolve().parents[1])
run_logged(["make", "all", "tests/kernel_validation", f"sm={SM}",
            f"NVCC={NVCC}", f"CUDADIR={cuda_dir}"], "gpu_build.txt")
run_logged(["python3", "scripts/run_tests.py"], "gpu_validation.txt")


## Measure GPU performance

The benchmark uses 100 timed repetitions after warm-up. Transfers and memory
allocations are excluded. It saves the raw program outputs, CSV measurements,
and environment metadata for 18 configurations. Above 1, the displayed speedup
means that the custom kernel was faster than cuBLAS in that particular case.


In [ ]:
run_logged(["python3", "scripts/benchmark.py"], "gpu_benchmark_run.txt")
import pandas as pd
csv_path = sorted(results.glob("benchmark_*.csv"))[-1]
display(pd.read_csv(csv_path))


## Optional Compute Sanitizer checks

Enable the next cell if Compute Sanitizer is installed in the runtime. The
checks run only against the custom-kernel validation executable. An unavailable
sanitizer is not a passed sanitizer check.


In [ ]:
RUN_SANITIZER = False
if RUN_SANITIZER:
    sanitizer = shutil.which("compute-sanitizer")
    if sanitizer is None:
        raise RuntimeError("Compute Sanitizer is not installed in this runtime.")
    for tool in ["memcheck", "racecheck", "synccheck"]:
        run_logged([sanitizer, "--tool", tool, "--error-exitcode", "1",
                    "./tests/kernel_validation"], f"sanitizer_{tool}.txt")
else:
    print("Sanitizer checks were not requested and have not been run.")


## Download the source and real results

The archive includes source and measured evidence. Add the new result files to
the public GitHub repository before submitting its URL through the homework
form. The saved notebook has no stored execution outputs by design; the result
logs preserve the actual run.


In [ ]:
from google.colab import files
output_zip = Path("/content/AKT26_HW4_GPU_Results.zip")
excluded = {"3-ger/ger", "4-gemm/gemm", "tests/kernel_validation"}
with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(repo.rglob("*")):
        if not path.is_file():
            continue
        relative = path.relative_to(repo)
        if ".git" in relative.parts or "__pycache__" in relative.parts:
            continue
        if relative.as_posix() in excluded:
            continue
        archive.write(path, "akt26_gpu_dev/" + relative.as_posix())
files.download(str(output_zip))
print("Saved source and measured results:", output_zip)
